# FarmTech Solutions — Sistema de Visão Computacional
## FIAP — Fase 6 | Redes Neurais & Visão Computacional

**RM:** 567029  
**Grupo:** FarmTech Solutions  

**Integrantes:**  
- William Albert Cesário Vasconcelos  
- Pedro Alves da Silva  
- Douglas Rafael do Amaral  
- Cláudio Sartori  

---

## Contexto do Projeto

A FarmTech Solutions está expandindo seus serviços de IA para além do agronegócio, atuando agora na área de **visão computacional**. Neste projeto, desenvolvemos um sistema capaz de detectar e classificar objetos em imagens usando três abordagens diferentes:

- **Entrega 1:** YOLOv5 customizado — treinado na nossa própria base de imagens
- **Entrega 2:** YOLO padrão pré-treinado (COCO) + CNN treinada do zero

### Objetos escolhidos

- **Classe 0:** Controle de PS4 (DualShock 4)
- **Classe 1:** Livro

Esses objetos foram escolhidos por serem **visualmente muito distintos** — formas, texturas e contextos de uso completamente diferentes, o que é ideal para demonstrar as capacidades do modelo com um dataset pequeno.

### Estrutura do Dataset

| Conjunto   | Controle PS4 | Livro      | Total |
|------------|-------------|------------|-------|
| Treino     | 32 imagens  | 32 imagens | 64    |
| Validação  | 4 imagens   | 4 imagens  | 8     |
| Teste      | 4 imagens   | 4 imagens  | 8     |
| **Total**  | **40**      | **40**     | **80**|

---
# ENTREGA 1 — YOLOv5 Customizado

## Etapa 1: Configuração do Ambiente

Montamos o Google Drive, clonamos o repositório oficial do YOLOv5 e instalamos todas as dependências necessárias.

In [ ]:
# Montar o Google Drive para acessar o dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clonar o repositório oficial do YOLOv5 e instalar dependências
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

In [ ]:
# Importar bibliotecas essenciais
import os
import shutil
import yaml
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print('Bibliotecas importadas com sucesso!')
print('Diretorio atual: ' + os.getcwd())

## Etapa 2: Organização do Dataset

Definimos os caminhos das pastas e criamos o arquivo `dataset.yaml`, que o YOLOv5 usa para localizar as imagens e identificar as classes.

> **Nota:** O nome da pasta no Drive contém um espaço no final (`fase6_yolo `), por isso o caminho inclui esse espaço.

In [ ]:
# Definir caminhos do dataset
DRIVE_BASE   = '/content/drive/MyDrive/fase6_yolo /dataset'
TRAIN_IMAGES = DRIVE_BASE + '/images/train'
VAL_IMAGES   = DRIVE_BASE + '/images/val'
TEST_IMAGES  = DRIVE_BASE + '/images/test'
TRAIN_LABELS = DRIVE_BASE + '/labels/train'
VAL_LABELS   = DRIVE_BASE + '/labels/val'

# Verificar se todas as pastas existem e contar arquivos
for path in [TRAIN_IMAGES, VAL_IMAGES, TEST_IMAGES, TRAIN_LABELS, VAL_LABELS]:
    n = len(os.listdir(path)) if os.path.exists(path) else 0
    status = 'OK' if n > 0 else 'ERRO'
    print('[%s] %s (%d arquivos)' % (status, path.split('dataset/')[-1], n))

In [ ]:
# Criar o arquivo dataset.yaml — necessario para o YOLOv5 encontrar as imagens
import yaml

dataset_yaml = {
    'path':  DRIVE_BASE,
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    2,
    'names': ['controle_ps4', 'livro']
}

yaml_path = '/content/yolov5/dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False, allow_unicode=True)

print('dataset.yaml criado!')
with open(yaml_path, 'r') as f:
    print(f.read())

## Etapa 3: Visualização do Dataset

Antes de treinar, verificamos visualmente as imagens para garantir que o dataset está correto.

In [ ]:
# Visualizar amostras do dataset de treino
train_imgs = (glob.glob(TRAIN_IMAGES + '/*.jpg') +
              glob.glob(TRAIN_IMAGES + '/*.jpeg') +
              glob.glob(TRAIN_IMAGES + '/*.png'))

print('Total de imagens de treino: %d' % len(train_imgs))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Amostras do Dataset de Treino', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    if i < len(train_imgs):
        img = mpimg.imread(train_imgs[i])
        ax.imshow(img)
        ax.set_title(os.path.basename(train_imgs[i])[:20], fontsize=7)
        ax.axis('off')
    else:
        ax.axis('off')

plt.tight_layout()
plt.show()

## Etapa 4: Treinamento — Simulação 1 (30 épocas)

Primeiro treinamento com **30 épocas**. Parâmetros utilizados:

- `--img 640`: imagens redimensionadas para 640x640 pixels
- `--batch 16`: 16 imagens processadas por iteração
- `--epochs 30`: 30 ciclos completos de aprendizado
- `--weights yolov5s.pt`: parte de um modelo pré-treinado (transfer learning)
- `--cache`: armazena imagens em RAM para acelerar o treino

In [ ]:
# SIMULAÇÃO 1 — Treinamento com 30 épocas
print('Iniciando Simulacao 1: 30 epocas...')

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 30 \
    --data /content/yolov5/dataset.yaml \
    --weights yolov5s.pt \
    --name exp_30ep \
    --cache

## Etapa 5: Treinamento — Simulação 2 (60 épocas)

Segundo treinamento com **60 épocas** — o dobro da simulação anterior. Permite comparar se mais épocas trazem ganho real de performance.

In [ ]:
# SIMULAÇÃO 2 — Treinamento com 60 épocas
print('Iniciando Simulacao 2: 60 epocas...')

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 60 \
    --data /content/yolov5/dataset.yaml \
    --weights yolov5s.pt \
    --name exp_60ep \
    --cache

## Etapa 6: Comparação dos Resultados — 30 vs 60 épocas

Comparamos as principais métricas das duas simulações:

- **Precision:** de tudo que o modelo disse ser da classe X, quanto realmente era?
- **Recall:** de tudo que era da classe X, quanto o modelo encontrou?
- **mAP@0.5:** principal métrica de detecção (IoU de 50%)
- **mAP@0.5:0.95:** métrica mais rigorosa, avalia em múltiplos limiares de IoU
- **box_loss / cls_loss:** erros de localização e classificação

In [ ]:
# Carregar os CSVs de resultado gerados pelo YOLOv5
import pandas as pd

results_30 = pd.read_csv('/content/yolov5/runs/train/exp_30ep/results.csv')
results_60 = pd.read_csv('/content/yolov5/runs/train/exp_60ep/results.csv')

results_30.columns = results_30.columns.str.strip()
results_60.columns = results_60.columns.str.strip()

print('Colunas disponiveis:')
print(results_30.columns.tolist())

In [ ]:
# Plotar comparacao das metricas principais
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Comparacao: 30 Epocas vs 60 Epocas', fontsize=16, fontweight='bold')

metricas = [
    ('metrics/precision',    'Precision'),
    ('metrics/recall',       'Recall'),
    ('metrics/mAP_0.5',      'mAP@0.5'),
    ('metrics/mAP_0.5:0.95', 'mAP@0.5:0.95'),
    ('val/box_loss',         'Val Box Loss'),
    ('val/cls_loss',         'Val Class Loss'),
]

for ax, (col, titulo) in zip(axes.flatten(), metricas):
    if col in results_30.columns:
        ax.plot(results_30[col], label='30 epocas', color='royalblue', linewidth=2)
    if col in results_60.columns:
        ax.plot(results_60[col], label='60 epocas', color='tomato', linewidth=2)
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Epoca')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/comparacao_30_vs_60.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo!')

In [ ]:
# Tabela resumo com os melhores valores de cada simulacao
metricas_resumo = [
    'metrics/precision', 'metrics/recall',
    'metrics/mAP_0.5',   'metrics/mAP_0.5:0.95'
]

resumo = {}
for col in metricas_resumo:
    if col in results_30.columns:
        resumo[col] = {
            '30 epocas': round(results_30[col].max(), 4),
            '60 epocas': round(results_60[col].max(), 4),
        }

df_resumo = pd.DataFrame(resumo).T
df_resumo.index = ['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95']

print('TABELA COMPARATIVA — Melhores valores de cada experimento:')
print('=' * 50)
print(df_resumo.to_string())

## Etapa 7: Detecções nas Imagens de Teste

Aplicamos os modelos treinados nas **8 imagens de teste** — imagens que nunca foram vistas durante o treinamento.

In [ ]:
# Deteccao com modelo de 60 epocas
!python detect.py \
    --weights /content/yolov5/runs/train/exp_60ep/weights/best.pt \
    --img 640 \
    --conf 0.25 \
    --source '/content/drive/MyDrive/fase6_yolo /dataset/images/test' \
    --name detect_60ep

# Deteccao com modelo de 30 epocas
!python detect.py \
    --weights /content/yolov5/runs/train/exp_30ep/weights/best.pt \
    --img 640 \
    --conf 0.25 \
    --source '/content/drive/MyDrive/fase6_yolo /dataset/images/test' \
    --name detect_30ep

print('Deteccoes concluidas!')

In [ ]:
# Visualizar resultados — modelo 60 epocas
detect_path = '/content/yolov5/runs/detect/detect_60ep/'
detected_imgs = (glob.glob(detect_path + '*.jpg') +
                 glob.glob(detect_path + '*.jpeg') +
                 glob.glob(detect_path + '*.png'))

print('Imagens detectadas: %d' % len(detected_imgs))

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Resultados das Deteccoes — Modelo 60 Epocas', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    if i < len(detected_imgs):
        img = mpimg.imread(detected_imgs[i])
        ax.imshow(img)
        ax.set_title('Imagem %d' % (i+1), fontsize=9)
        ax.axis('off')
    else:
        ax.axis('off')

plt.tight_layout()
plt.savefig('/content/deteccoes_60ep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva!')

## Análise e Conclusões — Entrega 1

### Resultados obtidos

| Métrica       | 30 épocas | 60 épocas |
|---------------|-----------|----------|
| Precision     | 93.6%     | 95.9%    |
| Recall        | 99.6%     | 100%     |
| mAP@0.5       | 99.5%     | 99.5%    |
| mAP@0.5:0.95  | 72.7%     | 84.0%    |

O modelo de 60 épocas apresentou resultados superiores, especialmente no mAP@0.5:0.95, que subiu de 72.7% para 84.0%. Isso indica que o modelo aprendeu a posicionar as bounding boxes com maior precisão. O box_loss e cls_loss continuaram caindo até as 60 épocas sem sinais de overfitting, confirmando que mais épocas foram benéficas neste caso.

**Pontos fortes:** os objetos escolhidos são visualmente distintos, facilitando a diferenciação. Com apenas 64 imagens de treino, o modelo demonstrou excelente generalização, detectando 100% dos objetos nas imagens de teste.

**Limitações:** o dataset pequeno pode limitar a generalização em cenários muito variados. Para produção, seria ideal expandir para 500+ imagens por classe.

---
# ENTREGA 2 — Comparação de Abordagens

## Abordagem 2.1 — YOLO Padrão (pré-treinado no COCO)

O YOLOv5 padrão é treinado no dataset COCO com 80 classes genéricas. Testamos se ele já reconhece nossos objetos nativamente, sem nenhuma customização.

In [ ]:
# Rodar YOLO padrao (sem customizacao) nas imagens de teste
print('Rodando YOLO padrao (COCO)...')

!python detect.py \
    --weights yolov5s.pt \
    --img 640 \
    --conf 0.25 \
    --source '/content/drive/MyDrive/fase6_yolo /dataset/images/test' \
    --name detect_yolo_padrao

print('Concluido!')

In [ ]:
# Visualizar resultados do YOLO padrao
detect_padrao = '/content/yolov5/runs/detect/detect_yolo_padrao/'
imgs_padrao = (glob.glob(detect_padrao + '*.jpg') +
               glob.glob(detect_padrao + '*.jpeg') +
               glob.glob(detect_padrao + '*.png'))

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('YOLO Padrao (COCO) — Deteccoes nas Imagens de Teste', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    if i < len(imgs_padrao):
        img = mpimg.imread(imgs_padrao[i])
        ax.imshow(img)
        ax.set_title('Imagem %d' % (i+1), fontsize=9)
        ax.axis('off')
    else:
        ax.axis('off')

plt.tight_layout()
plt.savefig('/content/deteccoes_yolo_padrao.png', dpi=150, bbox_inches='tight')
plt.show()

## Abordagem 2.2 — CNN Treinada do Zero

Construímos uma CNN do zero usando TensorFlow/Keras para **classificar** as imagens. A diferença fundamental em relação ao YOLO é que a CNN apenas diz "esta imagem é da classe X" — sem localizar o objeto com bounding box.

A arquitetura tem 3 blocos convolucionais progressivos (32 → 64 → 128 filtros), com BatchNormalization e Dropout para evitar overfitting, e Data Augmentation para ampliar artificialmente o dataset pequeno.

In [ ]:
# Organizar imagens em subpastas por classe para a CNN
# Estrutura necessaria: split/classe/imagem.jpg
import os, shutil, glob

CNN_BASE = DRIVE_BASE + '/cnn'

for split in ['train', 'val', 'test']:
    os.makedirs(CNN_BASE + '/' + split + '/controle_ps4', exist_ok=True)
    os.makedirs(CNN_BASE + '/' + split + '/livro', exist_ok=True)

# Copiar imagens para subpastas corretas baseado no horario no nome do arquivo
for split in ['train', 'val', 'test']:
    imagens = (glob.glob(DRIVE_BASE + '/images/' + split + '/*.jpg') +
               glob.glob(DRIVE_BASE + '/images/' + split + '/*.jpeg') +
               glob.glob(DRIVE_BASE + '/images/' + split + '/*.png'))
    for img_path in imagens:
        nome = os.path.basename(img_path).lower()
        if '18.30' in nome or '18:30' in nome:
            destino = CNN_BASE + '/' + split + '/livro/'
        else:
            destino = CNN_BASE + '/' + split + '/controle_ps4/'
        shutil.copy(img_path, destino)

for split in ['train', 'val', 'test']:
    for classe in ['controle_ps4', 'livro']:
        n = len(os.listdir(CNN_BASE + '/' + split + '/' + classe))
        print('%s/%s: %d imagens' % (split, classe, n))

In [ ]:
# Importar bibliotecas e criar os geradores de dados
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import time

tf.random.set_seed(42)
np.random.seed(42)

IMG_SIZE   = (128, 128)
BATCH_SIZE = 16

# Data Augmentation no treino para ampliar artificialmente o dataset
train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=15,
    width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True, zoom_range=0.1, fill_mode='nearest'
)
val_datagen  = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    CNN_BASE + '/train', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
val_generator = val_datagen.flow_from_directory(
    CNN_BASE + '/val', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    CNN_BASE + '/test', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

print('Classes: ' + str(train_generator.class_indices))
print('Treino: %d | Val: %d | Teste: %d' % (
    train_generator.samples, val_generator.samples, test_generator.samples))

In [ ]:
# Construir a arquitetura da CNN do zero
model = keras.Sequential([
    layers.Input(shape=(128, 128, 3)),

    # Bloco 1: detecta features simples (bordas, texturas)
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Bloco 2: detecta features intermediarias
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Bloco 3: detecta features complexas (formas, objetos)
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Cabeca classificadora
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(2, activation='softmax')  # saida: probabilidade por classe
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Treinar a CNN com EarlyStopping e ReduceLROnPlateau
callbacks = [
    keras.callbacks.EarlyStopping(
        patience=10, restore_best_weights=True, monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

inicio = time.time()
history = model.fit(
    train_generator, epochs=50,
    validation_data=val_generator,
    callbacks=callbacks, verbose=1
)
tempo = time.time() - inicio
print('Treinamento: %.1fs (%.1f min)' % (tempo, tempo/60))

In [ ]:
# Curvas de aprendizado da CNN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de Aprendizado — CNN do Zero', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'],     label='Treino',    color='royalblue')
axes[0].plot(history.history['val_accuracy'], label='Validacao', color='tomato')
axes[0].set_title('Acuracia'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Treino',    color='royalblue')
axes[1].plot(history.history['val_loss'], label='Validacao', color='tomato')
axes[1].set_title('Loss (Erro)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/curvas_cnn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Avaliar a CNN nas imagens de teste
inicio_inf = time.time()
predictions = model.predict(test_generator)
tempo_inf = time.time() - inicio_inf

y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print('Relatorio de Classificacao — CNN do Zero:')
print('=' * 55)
print(classification_report(y_true, y_pred, target_names=class_names))
print('Inferencia: %.3fs para %d imagens' % (tempo_inf, len(y_pred)))

# Matriz de confusao
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de Confusao — CNN do Zero')
plt.ylabel('Real'); plt.xlabel('Predicao')
plt.tight_layout()
plt.savefig('/content/matriz_confusao_cnn.png', dpi=150)
plt.show()

## Comparação Final das 3 Abordagens

In [ ]:
# Tabela comparativa final das 3 abordagens
import pandas as pd

comparacao = {
    'Criterio': [
        'Tarefa',
        'Facilidade de uso',
        'Precisao no teste',
        'Tempo de treino',
        'Tempo de inferencia',
        'Localiza o objeto?',
        'Dataset necessario',
    ],
    'YOLOv5 Customizado (60ep)': [
        'Deteccao + Classificacao',
        'Media (requer rotulacao)',
        'mAP@0.5: 99.5% | mAP@0.5:0.95: 84.0%',
        '0.028h (~2 min GPU)',
        '~8-10ms por imagem',
        'Sim (bounding box)',
        'Imagens + labels YOLO',
    ],
    'YOLO Padrao (COCO)': [
        'Deteccao + Classificacao',
        'Alta (zero configuracao)',
        'Nao reconhece controle_ps4',
        'Zero (pre-treinado)',
        '~8-10ms por imagem',
        'Sim (bounding box)',
        'Nao necessario',
    ],
    'CNN do Zero': [
        'Classificacao apenas',
        'Alta (Keras simples)',
        'Acuracia: 100% (8/8)',
        '46.8s (~1 min GPU)',
        '1.4s para 8 imagens',
        'Nao',
        'Imagens separadas por pasta',
    ]
}

df = pd.DataFrame(comparacao).set_index('Criterio')
print('TABELA COMPARATIVA FINAL')
print('=' * 80)
print(df.to_string())

## Conclusões Finais — Entrega 2

### YOLOv5 Customizado
Melhor abordagem quando o problema exige saber **onde** o objeto está na imagem. Com apenas 64 imagens de treino, atingiu 99.5% de mAP@0.5 e 100% de Recall. Indicado para segurança, controle de acesso e monitoramento em tempo real.

### YOLO Padrão (COCO)
Detecta bem objetos entre as 80 classes do COCO, mas não reconhece `controle_ps4`. Útil quando os objetos já estão no COCO e não há tempo para treinamento.

### CNN do Zero
Atingiu 100% de acurácia no teste em ~47 segundos de treinamento. Mais simples de implementar — não exige rotulação com bounding boxes. A limitação é que não fornece localização do objeto.

### Recomendação Final para o Cliente FarmTech

Para aplicações de **segurança patrimonial e controle de acesso** — onde é necessário detectar e localizar objetos em tempo real — o **YOLOv5 customizado é a melhor escolha**. A CNN do zero é uma excelente alternativa quando apenas a classificação é suficiente. O YOLO padrão só deve ser usado quando os objetos já pertencem às 80 classes do COCO.

In [ ]:
# Salvar todos os resultados no Google Drive
import shutil, os

OUTPUT = '/content/drive/MyDrive/fase6_yolo /resultados'
os.makedirs(OUTPUT, exist_ok=True)

arq1 = '/content/comparacao_30_vs_60.png'
arq2 = '/content/deteccoes_60ep.png'
arq3 = '/content/deteccoes_yolo_padrao.png'
arq4 = '/content/curvas_cnn.png'
arq5 = '/content/matriz_confusao_cnn.png'

if os.path.exists(arq1): shutil.copy(arq1, OUTPUT); print('Salvo: comparacao_30_vs_60.png')
if os.path.exists(arq2): shutil.copy(arq2, OUTPUT); print('Salvo: deteccoes_60ep.png')
if os.path.exists(arq3): shutil.copy(arq3, OUTPUT); print('Salvo: deteccoes_yolo_padrao.png')
if os.path.exists(arq4): shutil.copy(arq4, OUTPUT); print('Salvo: curvas_cnn.png')
if os.path.exists(arq5): shutil.copy(arq5, OUTPUT); print('Salvo: matriz_confusao_cnn.png')

shutil.copy('/content/yolov5/runs/train/exp_60ep/weights/best.pt',
            OUTPUT + '/best_60ep.pt')
print('Modelo best.pt salvo!')
print('Todos os resultados salvos em: ' + OUTPUT)